# Strategy Dimension: AI in the Business description (Item 1)

Extensive margin (`ai_sentence_share`) and intensive margin (`net_tone`, the
mean FinBERT-tone net sentiment P(positive) - P(negative) in [-1, 1]) of
AI-related sentences in Item 1 (Business) of the 10-K. Item 1 is where a firm
frames AI as part of what it does, so it captures strategic positioning.

Depends on the shared caches written by `nlp_features_setup.ipynb`.
Writes `data_clean/indicators/strategy.parquet`.

In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import logging
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.indicators.common.io import (
    cache_path,
    clear_cache,
    load_cached_step,
    save_cached_step,
)
from src.indicators.nlp_features import (
    DIMENSIONS,
    MIN_AI_SENTENCES_FOR_TONE,
    aggregate_dimension,
    score_forward_looking,
    score_sentences,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")
os.environ.setdefault("EDGAR_IDENTITY", "Timo Koba kab.timo3@gmail.com")

FORCE_REFRESH = False
SHARED = "nlp_features"
DIM = DIMENSIONS["strategy"]

# Load the shared front-end outputs produced by nlp_features_setup.ipynb.
filings = load_cached_step(SHARED, "filings")
sentences_df = load_cached_step(SHARED, "sentences")
sentence_totals_df = load_cached_step(SHARED, "sentence_totals")
if filings is None or sentences_df is None or sentence_totals_df is None:
    raise RuntimeError(
        "Shared caches not found. Run notebooks/02_feature_engineering/"
        "nlp_features_setup.ipynb first."
    )

item_sentences = sentences_df[sentences_df["item"] == DIM.item].reset_index(drop=True)
n_filings = item_sentences["accession_number"].nunique() if len(item_sentences) else 0
print(f"Dimension: {DIM.name}  |  Item: {DIM.item}  |  tone: {DIM.tone}")
print(f"AI sentences in {DIM.item}: {len(item_sentences)} across {n_filings} filings")

## Score AI sentences

Score the Item's AI sentences with the dimension's tone model
(FinBERT-tone for sentiment dimensions, FinBERT-FLS for the
forward-looking dimension). Per-sentence results are cached on disk by
`sha256(sentence)`, so re-runs are near-free. The scored slice is cached
under this dimension's own namespace.

In [ ]:
scored = None if FORCE_REFRESH else load_cached_step(DIM.name, "scored")
if scored is None:
    if len(item_sentences) == 0:
        scored = item_sentences.copy()
    elif DIM.tone == "sentiment":
        scores = score_sentences(item_sentences["sentence"].tolist())
        scored = pd.concat(
            [item_sentences, scores[["pos", "neu", "neg", "label", "confidence"]]],
            axis=1,
        )
    else:  # forward_looking
        scores = score_forward_looking(item_sentences["sentence"].tolist())
        scored = pd.concat(
            [item_sentences, scores[["p_specific", "p_nonspecific", "p_not", "label", "confidence"]]],
            axis=1,
        )
    save_cached_step(scored, DIM.name, "scored")
    print(f"Scored {len(scored)} {DIM.item} AI sentences (saved to {cache_path(DIM.name, 'scored')})")
else:
    print(f"Loaded {len(scored)} scored sentences from cache ({cache_path(DIM.name, 'scored')})")
scored.head()

## Aggregate to firm level and write the indicator parquet

One row per firm: the extensive margin (`ai_sentence_share`) plus the
dimension's intensive-margin tone, set to NaN below
`MIN_AI_SENTENCES_FOR_TONE` AI sentences (too few to compare tone across
firms). Each row also carries `parse_complete` (1 only if all three Items
parsed for the filing); it is identical across the three dimensions, so
filter `parse_complete == 1` for a sample consistent across dimensions, or
filter on this Item's own `n_total_sentences` to keep every firm valid for
this dimension. Written to `data_clean/indicators/<dimension>.parquet`.

In [ ]:
indicator = aggregate_dimension(DIM, filings, scored, sentence_totals_df)
tone_col = "net_tone" if DIM.tone == "sentiment" else "fls_score"
print(f"{DIM.name}: {len(indicator)} firm rows -> data_clean/indicators/{DIM.name}.parquet")
print(f"Firms with AI mention in {DIM.item}:        {int(indicator['has_ai_mention'].sum())}")
print(f"Firms above tone threshold (>= {MIN_AI_SENTENCES_FOR_TONE}):  {int(indicator[tone_col].notna().sum())}")
indicator.head()

## Sanity checks

Coverage, the intensive-margin tone summary, and the tone-threshold
pass-rate at 3/5/10/20 AI sentences. Use the pass-rate to revisit
`MIN_AI_SENTENCES_FOR_TONE` in `aggregate.py` empirically: each dimension
now uses a single Item, so pass-rates are lower than the old pooled
indicator.

In [ ]:
n_with_ai = int(indicator["has_ai_mention"].sum())
print("=== Coverage ===")
print(f"  total firms:             {len(indicator)}")
print(f"  with AI mention:         {n_with_ai}  ({indicator['has_ai_mention'].mean():.2%})")
print(f"  ai_sentence_share mean:  {indicator['ai_sentence_share'].mean():.4f}  sd: {indicator['ai_sentence_share'].std():.4f}")

if n_with_ai > 0:
    if DIM.tone == "sentiment":
        sub = indicator.loc[indicator["net_tone"].notna(), "net_tone"]
        print(f"\n=== Sentiment tone (>= {MIN_AI_SENTENCES_FOR_TONE} AI sentences) ===")
        print(f"  firms scored: {len(sub)}  ({len(sub) / n_with_ai:.1%} of AI-mentioning)")
        if len(sub):
            print(f"  net_tone mean: {sub.mean():+.3f}  sd: {sub.std():.3f}")
    else:
        sub = indicator.loc[indicator["fls_score"].notna(), "fls_score"]
        print(f"\n=== Forward-looking score (>= {MIN_AI_SENTENCES_FOR_TONE} AI sentences) ===")
        print(f"  firms scored: {len(sub)}  ({len(sub) / n_with_ai:.1%} of AI-mentioning)")
        if len(sub):
            print(f"  fls_score mean: {sub.mean():.3f}  sd: {sub.std():.3f}")

    print("\n=== Tone-threshold diagnostic ===")
    n_ai = indicator["n_ai_sentences"]
    for thr in (3, 5, 10, 20):
        n_pass = int((n_ai >= thr).sum())
        print(f"  n_ai_sentences >= {thr:>2}: {n_pass:>3} firms  ({n_pass / n_with_ai:.1%} of AI-mentioning)")

    print("\n=== n_ai_sentences distribution (AI-mentioning firms) ===")
    print(n_ai[n_ai > 0].describe().to_string())
else:
    print("no firms with AI mentions in this item")